# Co je to Nanopore sekvenování?

## Jak to funguje?

- Nanopór je maličký otvor v tenké membráně.
- Skrz nanopór protéká elektrolyt a tím i slabý elektrický proud.
- Když DNA prochází skrz pór, **každý DNA nukleotid (A, T, C, G)** trochu změní proud. Každý jiným způsobem.
- Tato změna se zaznamenává jako **průběh signálu**.
- Počítač pak z tohoto signálu pozná, jaká písmena DNA šla dovnitř.

Podívej se na krátké video (2 minuty):  
🔗 [Jak funguje nanopórové sekvenování – od Oxford Nanopore](https://www.youtube.com/watch?v=VxGliKyYuFQ)

## Podíváme se na skutečný signál

V následujícím grafu uvidíme, jak vypadá **surový signál** ze sekvenátoru.

In [ ]:
import pod5
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.offline as pyo
pyo.init_notebook_mode()

# načteme ready z .pod5 souboru
file_path = "example.pod5"
with pod5.Reader(file_path) as reader:
    read = next(reader.reads())
    signal = read.signal
    sample_rate = read.run_info.sample_rate
    time_s = np.arange(len(signal)) / sample_rate # vteriny vztazene k zacatku cteni

# vykreslíme surový signál
plt.figure(figsize=(14, 4))
plt.plot(signal[1000:1500])  # Plot first 300 signal points
plt.title("Hrubý signál jednoho čtení z Nanopore")
plt.xlabel("Čas [s]")
plt.ylabel("Proud [pA]")
plt.grid(True)
plt.show()

### Otázky na zamyšlení:

**Proč se mění signál, když DNA prochází nanopórem?**

<details>
<summary>Odpověď</summary>

Každé písmeno DNA má jiný tvar a jinou velikost, takže ovlivňuje proud jinak. Když DNA prochází nanopórem, ovlivňuje proud iontů (např. Na⁺, Cl⁻), který jím prochází. Různé kombinace písmen DNA (tzv. k-mery) mají různý tvar a elektrické vlastnosti, takže každý z nich mění proud jiným způsobem. Počítač pak z těchto změn pozná, která písmena tam byla.

![DNA nukleotidy](images/dna-nucleotides.png)
</details>

**Když dostaneme tento signál, jak bys z něj poznal, jaká písmena jsou v DNA?**

<details>
<summary>Odpověď</summary>
Počítač se to učí z tisíců příkladů pomocí umělé inteligence.

**Zkus si představit:**
Kdybys měl z poslechu hluku určit, co jede po silnici (auto, kolo, autobus), dokážeš to? Nanopór to dělá podobně – rozpoznává, co právě prošlo.
</details>

## Převod signálu na DNA sekvenci – Basecalling

Když molekula DNA prochází nanopórem, zaznamenáváme změny elektrického proudu. Tyto změny tvoří tzv. **surový signál** (anglicky *raw signal*), který je uložen v souborech typu `.pod5`.

Ale abychom zjistili, jaká písmena DNA (A, T, C, G) odpovídají tomuto signálu, potřebujeme **převodník** – ten se nazývá **basecaller**.

## Co dělá basecaller?

- Analyzuje průběh elektrického signálu
- Pomocí umělé inteligence a trénovaných modelů odhadne, jaké nukleotidy procházely nanopórem
- Výsledek je sekvence DNA ve formátu **FASTQ** – tedy:
  - Sekvence (např. `AGGCTTAC...`)
  - Kvalita každého písmena (číslem vyjádřená jistota)

Pro převod použijeme nástroj **Dorado** od Oxford Nanopore Technologies.

V další části nejprve ukážeme basecalling jednotlivých čtení v jednom .pod5 souboru, a poté zpracujeme všechna najednou.

In [ ]:
# !python3 print_basecalled.py /home/ont/Desktop/noc_vedcu/dna_r10.4.1_e8.2_400bps_fast@v5.2.0 example.pod5

### Co je ve FASTQ souboru?

Soubor `.fastq` obsahuje výsledky sekvenování – tedy čtení DNA. Každé čtení má 4 řádky:

1. `@` a název čtení
2. Sekvence DNA (např. `ACGTTG...`)
3. `+` jako oddělovač
4. Kvalita jednotlivých bází (čím vyšší ASCII znak, tím vyšší jistota) - [tabulka vysvětlující  Phred skóre](https://en.wikipedia.org/wiki/Phred_quality_score)

In [ ]:
!zcat example.fastq.gz  | head -n 24

### Zkus si: K jakému organismu patří náš vzorek?

1. Vyber jednu ze sekvencí výše
2. Zkopíruj ji a vlož do nástroje [NCBI BLAST (nucleotide)](https://blast.ncbi.nlm.nih.gov/Blast.cgi?PAGE_TYPE=BlastSearch&PROGRAM=blastn)
3. Stiskni "BLAST"
4. Sleduj, jak databáze najde podobné sekvence a řekne ti, odkud pochází

### Zamysli se

- Jsou některé čtení delší než jiné?
- Všimni si, že některá čtení mají horší kvalitu (nižší znaky ve 4. řádku), použij tabulku
- Co se asi stane, když pošleme do BLASTu krátké vs. dlouhé čtení?

<details>
<summary>Odpověď</summary>

- Ano, délka čtení se může lišit. Nanopore umožňuje číst velmi dlouhé úseky DNA, ale některá čtení mohou být přerušena dříve.
- Znaky ve 4. řádku ukazují kvalitu každé báze. Nižší znaky znamenají nižší jistotu – mohou být způsobeny šumem v signálu.
- Krátká čtení mají menší šanci najít jednoznačný výsledek v BLASTu – mohou se hodit k více oblastem v různých genomech. Delší čtení jsou často specifičtější a dají lepší nápovědu o původu vzorku.

</details>

## Jak se liší genetická informace zdravého člověka a onkopacienta?

### Co je to karyotyp a proč ho zkoumáme?

Každá lidská buňka (kromě pohlavních buněk) obsahuje 46 chromozomů – to je náš karyotyp. Chromozomy jsou "balíčky" DNA, které nesou naši genetickou informaci. Zdravé buňky mají kompletní a správně uspořádaný karyotyp – 23 párů chromozomů, žádné navíc, žádné chybějící části.

U nádorových buněk ale často dochází ke změnám – mohou mít některé chromozomy zdvojené, ztracené, přeskupené nebo přerušené. Takové změny označujeme jako chromozomální aberace, a jsou jedním z typických znaků rakovinných buněk. Právě proto se karyotyp nádorových buněk zkoumá – může nám napovědět, co je v buňce špatně a jak rakovina vznikla.

<img src="images/human_male_karyotype.png" alt="drawing" width="700"/>

### Jak to zjistíme pomocí sekvenování

Pomocí nanopórového sekvenování DNA získáme dlouhé náhodně vybrané úseky genetické informace z jednotlivých chromozomů. Výsledkem je obrovské množství "čtení" – krátkých úseků DNA, které se zpětně zarovnají na jednotlivé chromozomy lidského genomu.

Při analýze sledujeme tzv. pokrytí (coverage) – tedy kolik čtení se mapuje na jednotlivé části každého chromozomu. Pokud má nějaký chromozom vyšší pokrytí, může to znamenat, že je v buňce zdvojený. Naopak nižší pokrytí může naznačovat ztrátu části nebo celého chromozomu.

### Proč porovnáváme zdravé a nádorové buňky?

Porovnáním karyotypu zdravých pacientů a onkopacientů můžeme zjistit, které chromozomy nebo jejich části jsou v nádorových buňkách změněné. To nám pomáhá pochopit, co se v buňkách děje při vzniku rakoviny, a může to být i důležitý krok ke zlepšení diagnostiky nebo léčby.

### Jak získáme pokrytí jednotlivých chromozomů?

Abychom mohli porovnat karyotyp zdravého a nemocného člověka, potřebujeme nejprve jejich sekvenovanou DNA zarovnat na referenční lidský genom – tedy na "mapu" správně uspořádané lidské DNA, kterou vědci sestavili.

Zarovnání na referenci: Pomocí bioinformatických nástrojů (např. minimap2) zarovnáme všechna přečtení DNA (tzv. "reads") ke správnému místu v lidském genomu.


In [ ]:
# TODO: uprav cesty a vzory souborů podle letošních dat
# Pro každý vzorek: kde leží jeho FASTQ soubory a do jaké skupiny patří
# (skupina = healthy_control / cancer_sample1 / cancer_sample2 - podle skupiny se počítají průměry a ANOVA)
test1 = "example.fastq.gz"
test2 = "example2.fastq.gz"
test3 = "example3.fastq.gz"

SAMPLES = {
    "healthy_control_rep1": {"fastq_glob": test1, "group": "healthy_control"},
    "healthy_control_rep2": {"fastq_glob": test1, "group": "healthy_control"},
    "healthy_control_rep3": {"fastq_glob": test1, "group": "healthy_control"},
    "cancer_sample1_rep1":  {"fastq_glob": test2, "group": "cancer_sample1"},
    "cancer_sample1_rep2":  {"fastq_glob": test2, "group": "cancer_sample1"},
    "cancer_sample1_rep3":  {"fastq_glob": test2, "group": "cancer_sample1"},
    "cancer_sample2_rep1":  {"fastq_glob": test3, "group": "cancer_sample2"},
    "cancer_sample2_rep2":  {"fastq_glob": test3, "group": "cancer_sample2"},
    "cancer_sample2_rep3":  {"fastq_glob": test3, "group": "cancer_sample2"},
}

reference_genome = "Homo_sapiens.GRCh38.dna.toplevel.fa"

In [ ]:
from workshop_helpers import *

# spojí FASTQ soubory, zarovná na referenční genom a vyfiltruje kvalitně namapovaná čtení
bams = merge_and_align(SAMPLES, reference_genome)

#### Jak vypadají namapovaná čtení?

In [ ]:
# SPUST IGV DESKTOP a vyber referenční lidský genom ze souboru (stažený v tomto adresáři)
# !igv

Výpočet pokrytí (coverage): Poté spočítáme, kolik čtení připadá na každý chromozom nebo jeho část. Čím více přečtení na určitém místě, tím vyšší pokrytí.

Vykreslení grafu: Pokrytí zobrazíme v grafu – každý chromozom bude mít svůj sloupec nebo bod. Zdravý vzorek by měl mít zhruba rovnoměrné pokrytí všech chromozomů. Nádorový vzorek ale může mít výrazné výkyvy – některé chromozomy s vyšším nebo nižším pokrytím, což ukazuje na změny v počtu kopií chromozomů (copy number variations).

## Vykreslíme a porovnáme pokrytí jednotlivých chromozomů

Porovnáme pokrytí celých chromozomů.

In [ ]:
# spočítáme pokrytí jednotlivých chromozomů pro každý vzorek
coverages_df, read_counts = compute_genome_coverage(bams)
coverages_df

#### Normalizace hloubky sekvenování

Po namapování čtení na referenční genom můžeme spočítat, kolikrát byla každá část DNA přečtena — tomu říkáme **pokrytí** (*coverage*). Ale...

##### Proč musíme pokrytí normalizovat?

Každý vzorek (nebo čárový kód) může mít **různý počet čtení**. Například:

- Vzorek A: 10 000 čtení
- Vzorek B: 100 000 čtení

Pokud bychom porovnávali jejich pokrytí přímo, zdálo by se, že vzorek B má "více DNA" — ale to je jen proto, že jsme ho více sekvenovali.

Abychom mohli porovnávat vzorky **spravedlivě**, musíme jejich pokrytí **přepočítat na jedno čtení**.

##### Jak to uděláme?

📌 Pro každý vzorek:
1. Spočítáme **pokrytí** (např. kolikrát byla přečtena každá část chromozomu).
2. Spočítáme, kolik bylo celkem čtení zarovnaných na celý genom.
3. Vydělíme pokrytí počtem čtení → tím dostaneme **normalizované pokrytí**.

Díky tomu můžeme porovnat, jestli má některý chromozom (nebo část genomu) vyšší nebo nižší pokrytí **nezávisle na tom, kolik čtení bylo celkem**.

To je užitečné například pro hledání **chromozomových aberací** — když je nějaký chromozom přítomen vícekrát nebo chybí.

In [ ]:
# normalizujeme podle hloubky sekvenování a zprůměrujeme přes replikáty v rámci skupiny
coverages_df_normalized, coverages_mean_df = normalize_and_average(coverages_df, read_counts, SAMPLES)
coverages_mean_df

In [ ]:
plot_genome_overview(coverages_mean_df)

A nyní se podíváme na pokrytí zblízka.

Spočítáme pokrytí v oknech o velikosti 50 000 bází.

In [ ]:
# spočítáme normalizované pokrytí v oknech po 50 000 bázích, abychom se mohli podívat na pokrytí zblízka
depths_df_normalized = compute_windowed_coverage(bams, reference_genome, read_counts, window=50_000)
depths_df_normalized

In [ ]:
chrom_options = [str(c) for c in range(1, 23)] + ["X", "Y"]

Vyber si chromozom z nabídky a podívej se, jak se liší pokrytí mezi zdravými vzorky a vzorky nádoru. Zkus např. chromozomy **8** a **17** - ty bývají v nádorových buňkách často pozměněné.


In [ ]:
import ipywidgets as widgets
from ipywidgets import *

interact(lambda chrom_num: plot_chromosome_coverage(chrom_num, depths_df_normalized), chrom_num=chrom_options);

## Porovnání průměrného pokrytí mezi více vzorky – ANOVA

V grafech jsme už viděli **normalizované pokrytí** – to nám pomohlo vzorky vizuálně srovnat.  
Teď ale chceme **statisticky ověřit**, jestli se určité chromozomy mezi vzorky liší víc, než bychom čekali náhodou.

📌 **Pro tento test používáme přímo počty čtení** z mapování, ne normalizované hodnoty.

Teď máme pro každý chromozom změřené **pokrytí** (coverage) z více vzorků, a každý vzorek má několik biologických opakování (replikáty).  
Chceme zjistit, jestli se **průměrné pokrytí** mezi vzorky **významně liší**.

## Jak to funguje
- **ANOVA** (Analysis of Variance) je metoda, která porovnává průměry mezi třemi a více skupinami najednou.
- V našem případě:
  - Skupiny = jednotlivé vzorky (např. zdravá kontrola, rakovina vzorek 1, rakovina vzorek 2)
  - Hodnoty v každé skupině = pokrytí z jednotlivých replikátů
- Test nám dá dvě čísla:
  - **F-statistika** – ukazuje, jak velké jsou rozdíly mezi skupinami ve srovnání s rozdíly uvnitř skupin
  - **p-hodnota** – říká, jak pravděpodobné je, že takové rozdíly vznikly náhodou

## Jak číst výsledek
- **p < 0,05** → rozdíly mezi skupinami jsou pravděpodobně skutečné
- **p ≥ 0,05** → rozdíly mohou být jen náhoda

💡 *V našem kontextu*:  
Pokud ANOVA ukáže významný rozdíl u konkrétního chromozomu, může to znamenat, že v některém vzorku je daný chromozom zastoupen víc nebo míň než v ostatních – např. kvůli změně počtu kopií chromozomu.


In [ ]:
# pro každý chromozom porovnáme pokrytí mezi skupinami (ANOVA)
results_df, significant_df = run_anova(coverages_df, SAMPLES)

print("Výsledky ANOVA (p < 0.05):")
display(significant_df)

Pro úplnost, tady je tabulka výsledků pro úplně všechny chromozomy (i ty bez významného rozdílu):


In [ ]:
results_df